# 🧠 Non-Maximum Suppression (NMS)

Welcome to the hands-on explanation notebook for **Non-Maximum Suppression (NMS)**! In this notebook, we will:
1. Explain why NMS is critical for post-processing in object detection models (like YOLO).
2. Implement the standard NMS algorithm from scratch using NumPy.
3. Set up concrete bounding box cases with overlapping and non-overlapping predictions.
4. Apply NMS to filter out redundant detections.
5. Plot the original candidate boxes and highlights of kept vs. suppressed boxes using Matplotlib.
6. Introduce **Soft-NMS** as an advanced solution to handle overlapping distinct objects.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

## 1. Scratch NMS Implementation in NumPy

Here is the standard hard NMS algorithm implemented using vectorised NumPy operations. It iterates through the boxes sorted by confidence score, greedily selects the box with the highest score, and suppresses any other boxes that overlap with it above a specified threshold.

In [ ]:
def nms(boxes, scores, iou_threshold):
    if len(boxes) == 0:
        return []
        
    x1 = boxes[:, 0]
    y1 = boxes[:, 1]
    x2 = boxes[:, 2]
    y2 = boxes[:, 3]

    areas = (x2 - x1) * (y2 - y1)
    order = scores.argsort()[::-1]
    
    keep = []
    while order.size > 0:
        i = order[0]
        keep.append(i)
        
        if order.size == 1:
            break
            
        xx1 = np.maximum(x1[i], x1[order[1:]])
        yy1 = np.maximum(y1[i], y1[order[1:]])
        xx2 = np.minimum(x2[i], x2[order[1:]])
        yy2 = np.minimum(y2[i], y2[order[1:]])
        
        w = np.maximum(0.0, xx2 - xx1)
        h = np.maximum(0.0, yy2 - yy1)
        intersection = w * h
        
        union = areas[i] + areas[order[1:]] - intersection
        iou = intersection / union
        
        inds = np.where(iou <= iou_threshold)[0]
        order = order[inds + 1]
        
    return keep

## 2. Defining Bounding Box Predictions

We set up 5 candidate bounding boxes with their corresponding confidence scores. Some boxes represent highly overlapping predictions for the same object, while others represent distinct objects.

In [ ]:
boxes = np.array([
    [100.0, 100.0, 200.0, 200.0],  # Box 0 (score: 0.90)
    [105.0, 105.0, 205.0, 205.0],  # Box 1 (score: 0.80) - highly overlaps Box 0
    [100.0, 100.0, 150.0, 150.0],  # Box 2 (score: 0.40) - low overlap Box 0
    [300.0, 300.0, 400.0, 400.0],  # Box 3 (score: 0.95)
    [310.0, 310.0, 410.0, 410.0]   # Box 4 (score: 0.85) - highly overlaps Box 3
])
scores = np.array([0.90, 0.80, 0.40, 0.95, 0.85])
iou_threshold = 0.5

keep_indices = nms(boxes, scores, iou_threshold)
print("Kept indices:", [int(x) for x in keep_indices])

## 3. Visualizing NMS Results

Let's visualize the boxes. Green/solid boxes represent the kept detections, and red/dashed boxes represent the suppressed detections.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.set_xlim(50, 450)
ax.set_ylim(450, 50)  # Inverted y-axis to match typical image coordinate system

for idx in range(len(boxes)):
    box = boxes[idx]
    w = box[2] - box[0]
    h = box[3] - box[1]
    score = scores[idx]
    
    if idx in keep_indices:
        rect = patches.Rectangle((box[0], box[1]), w, h, linewidth=3, edgecolor='green', facecolor='none')
        ax.add_patch(rect)
        ax.text(box[0] + 5, box[1] + 20, f"Box {idx} ({score:.2f}) [KEPT]", color='green', fontweight='bold', fontsize=10)
    else:
        rect = patches.Rectangle((box[0], box[1]), w, h, linewidth=2, edgecolor='red', linestyle='--', facecolor='none', alpha=0.6)
        ax.add_patch(rect)
        ax.text(box[0] + 5, box[1] + 20, f"Box {idx} ({score:.2f}) [SUPPRESSED]", color='red', fontsize=10)

ax.set_title("Non-Maximum Suppression (NMS) Results (IoU threshold = 0.5)", fontsize=14)
ax.grid(True, linestyle='--', alpha=0.5)
plt.show()

## 4. Soft-NMS: Handling Occlusions and Side-by-Side Objects

Hard NMS completely discards overlapping bounding boxes if their IoU exceeds the threshold. In scenarios where objects are crowded (e.g. side-by-side components, pedestrians walking close to each other), this will lead to missed detections (False Negatives).

**Soft-NMS** decays the detection score of overlapping boxes as a continuous function of overlap, instead of instantly dropping them to zero. The score update rule is:
$$s_i \leftarrow s_i \cdot \exp\left( -\frac{\text{IoU}(M, b_i)^2}{\sigma} \right)$$

If the decayed score remains higher than a second score threshold, the object is preserved! This helps object detectors achieve much better recall in dense environments.